In [ ]:
# Общие импорты и локальные данные JERICHON
import os, sys
from pathlib import Path

_HERE = Path(globals().get("__vsc_ipynb_file__", Path.cwd())).expanduser().resolve()
_CANDIDATES = [_HERE, *_HERE.parents, Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next((p for p in dict.fromkeys(_CANDIDATES)
                     if (p / "src" / "data" / "load_data.py").is_file()
                     and (p / "research").is_dir()), None)
if PROJECT_ROOT is None and os.environ.get("JERICHON_PROJECT_ROOT"):
    PROJECT_ROOT = Path(os.environ["JERICHON_PROJECT_ROOT"]).expanduser().resolve()
if PROJECT_ROOT is None or not (PROJECT_ROOT / "src" / "data" / "load_data.py").is_file():
    raise FileNotFoundError("Не найден корень JERICHON с src/data/load_data.py")
sys.path[:0] = [str(PROJECT_ROOT), str(PROJECT_ROOT / "data")]
from src.data.load_data import *
ROOT = PROJECT_ROOT


# JERICHON — Super Candles: группировка признаков и event study v2

**Цель:** проверить источники и семейства Super Candle-признаков без 401 отдельных тестов.

Зафиксированные архитектурные правила из диагностики: Super Candles = только `tradestats`, `obstats`, `orderstats`; `orderbook` отдельно. Не делать простой join по `ts`: использовать существующий causal pipeline с валидированным `available_at <= bar_ts` и backward join. `SYSTIME` нельзя глобально считать UTC или MSK без проверки.

Этот notebook повторно строит локальный causal dataset, сворачивает трансформации признаков в корневые сигналы, формирует source/family scores и объединяет соседние срабатывания в эпизоды. На диск записывается **ровно один CSV**: `SC_02_group_research_summary.csv`. Результаты разведочные, не доказательство торгового преимущества.

## 1. Конфигурация

Используется локальная история AlgoPack, без API-запросов. Порог Dollar Bars должен совпадать с предыдущим исследованием.

In [7]:
from pathlib import Path
import sys, inspect, warnings, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TICKERS = ["SMLT", "GAZP", "MTLR", "VTBR"]
DOLLAR_THRESHOLD = 10_000_000
ROLLING_WINDOW = 500
MIN_HISTORY = 100
ANOMALY_QUANTILES = (0.95, 0.99)
GROUP_QUANTILES = (0.95, 0.99)
EPISODE_GAP_BARS = 3
MAX_CONTROLS_PER_EVENT = 3
RANDOM_STATE = 17
PROJECT_ROOT = ROOT
DATA_ROOT = PROJECT_ROOT / "data"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.builders import jerichon_feature_pipeline as jfp

required_helpers = ("build_dollar_bars_from_trades", "_base", "_build_sc_features", "_choose_availability", "_attach_source")
missing = [x for x in required_helpers if not hasattr(jfp, x)]
if missing:
    raise RuntimeError(f"В установленном jerichon_feature_pipeline.py отсутствуют helpers: {missing}")

print("Project root:", PROJECT_ROOT)
print("History root:", HISTORY_ROOT)
print("Pipeline:", Path(inspect.getfile(jfp)).resolve())
print("Tickers:", TICKERS)
print("Dollar threshold:", f"{DOLLAR_THRESHOLD:,}")


Project root: C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON
History root: C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data\for_ml\storage\history
Pipeline: C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data\for_ml\builders\jerichon_feature_pipeline.py
Tickers: ['SMLT', 'GAZP', 'MTLR', 'VTBR']
Dollar threshold: 10,000,000


## 2. Построение причинного набора

Повторно используем проверенную загрузку и существующий feature pipeline; контракт 73 + 401 = 474 проверяется до анализа.

In [8]:

def resolve_feature_frame(source, frames):
    if source in frames:
        return frames[source]
    matches = [k for k in frames if source.lower() in str(k).lower()]
    if len(matches) == 1:
        return frames[matches[0]]
    raise RuntimeError(f"Не удалось однозначно сопоставить {source}; доступны: {list(frames)}")

def build_ticker_dataset(ticker):
    data, saved_dates = load_local_history(ticker, HISTORY_ROOT)
    bars = jfp.build_dollar_bars_from_trades(data["trades"], threshold=DOLLAR_THRESHOLD)
    if bars is None or bars.empty:
        raise RuntimeError(f"{ticker}: Dollar Bar builder вернул пустой результат.")
    if "bar_ts" not in bars.columns:
        if "close_time" not in bars.columns:
            raise RuntimeError(f"{ticker}: в bars нет bar_ts/close_time; columns={list(bars.columns)}")
        bars["bar_ts"] = pd.to_datetime(bars["close_time"], errors="coerce", utc=True)
    else:
        bars["bar_ts"] = pd.to_datetime(bars["bar_ts"], errors="coerce", utc=True)
    bars = bars.dropna(subset=["bar_ts"]).sort_values("bar_ts").drop_duplicates("bar_ts", keep="last").reset_index(drop=True)

    # The pipeline creates many derived columns by repeated assignment. Its
    # fragmentation PerformanceWarnings are known and are summarized once,
    # rather than flooding the notebook output.
    perf_warning_count = 0
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always", pd.errors.PerformanceWarning)
        base = jfp._base(bars.copy())
    perf_warning_count += sum(issubclass(w.category, pd.errors.PerformanceWarning) for w in captured)
    if isinstance(base, tuple):
        base = base[0]
    if not isinstance(base, pd.DataFrame):
        raise TypeError(f"{ticker}: _base не вернул DataFrame")
    if "bar_ts" not in base.columns:
        if "close_time" in base.columns:
            base = base.rename(columns={"close_time": "bar_ts"})
        elif isinstance(base.index, pd.DatetimeIndex):
            base = base.reset_index().rename(columns={base.index.name or "index": "bar_ts"})
        elif "ts" in base.columns:
            base = base.rename(columns={"ts": "bar_ts"})
        else:
            raise RuntimeError(f"{ticker}: не найден временной ключ в base features")
    base["bar_ts"] = pd.to_datetime(base["bar_ts"], errors="coerce", utc=True)
    base = base.dropna(subset=["bar_ts"]).sort_values("bar_ts").drop_duplicates("bar_ts", keep="last")

    bars_sc = bars.copy()
    if "ts" not in bars_sc.columns:
        bars_sc["ts"] = bars_sc["bar_ts"]
    if "CLOSE" not in bars_sc.columns:
        close_col = next((c for c in bars_sc.columns if str(c).lower() == "close"), None)
        if close_col is None:
            raise RuntimeError(f"{ticker}: нет CLOSE/close в Dollar Bars")
        bars_sc["CLOSE"] = bars_sc[close_col]
    bars_sc["ts"] = pd.to_datetime(bars_sc["ts"], errors="coerce", utc=True)
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always", pd.errors.PerformanceWarning)
        sc_result = jfp._build_sc_features(
            data,
            bars_sc.dropna(subset=["ts"]).sort_values("ts").reset_index(drop=True),
        )
    perf_warning_count += sum(issubclass(w.category, pd.errors.PerformanceWarning) for w in captured)
    feature_frames = sc_result[0] if isinstance(sc_result, tuple) else sc_result
    if not isinstance(feature_frames, dict):
        raise TypeError(f"{ticker}: _build_sc_features должен вернуть dict")

    causal = bars.copy()
    for source in ("tradestats", "obstats", "orderstats"):
        # Validate source availability using the pipeline's established helper.
        checked = jfp._choose_availability(data[source].copy(), source)
        if isinstance(checked, tuple):
            checked = checked[0]
        if checked is None or len(checked) == 0:
            raise RuntimeError(f"{ticker}: availability validation вернула пустой {source}")
        ff = resolve_feature_frame(source, feature_frames)
        attached = jfp._attach_source(causal, ff, data[source], source)
        if isinstance(attached, pd.DataFrame):
            causal = attached

    base_cols_raw = [c for c in base.columns if c != "bar_ts" and pd.api.types.is_numeric_dtype(base[c])]
    if len(base_cols_raw) != 73:
        raise RuntimeError(f"{ticker}: ожидалось 73 base features, найдено {len(base_cols_raw)}")
    base_join = base[["bar_ts"] + base_cols_raw].copy()
    base_join = base_join.rename(columns={c: f"base__{c}" for c in base_cols_raw})
    causal["bar_ts"] = pd.to_datetime(causal["bar_ts"], errors="coerce", utc=True)
    causal = causal.merge(base_join, on="bar_ts", how="left", validate="one_to_one")

    provenance = [c for c in causal.columns if c.endswith("__event_ts") or c.endswith("__available_at")]
    prefixes = ("tradestats__", "obstats__", "orderstats__")
    sc_cols = [c for c in causal.columns if c.startswith(prefixes) and c not in provenance and pd.api.types.is_numeric_dtype(causal[c])]
    # IMPORTANT: test the dtype of the SERIES causal[c], not the column-name
    # string c. Testing c silently dropped all 73 base features.
    base_feature_cols = [
        c for c in causal.columns
        if c.startswith("base__") and pd.api.types.is_numeric_dtype(causal[c])
    ]
    feature_cols = base_feature_cols + sc_cols

    # Validate each block separately so a mismatch reports exactly which block
    # is absent; don't silently skip a ticker or hide the root cause.
    if len(base_feature_cols) != len(base_cols_raw):
        raise RuntimeError(
            f"{ticker}: после merge сохранилось {len(base_feature_cols)} из "
            f"{len(base_cols_raw)} базовых признаков"
        )
    if not sc_cols:
        raise RuntimeError(
            f"{ticker}: после causal attach не найдено числовых SC-признаков. "
            f"Колонки итогового frame: {list(causal.columns)}"
        )
    if len(feature_cols) != 474:
        raise RuntimeError(
            f"{ticker}: состав признаков отличается от ожидаемого: "
            f"base={len(base_feature_cols)} (ожидалось 73), "
            f"SC={len(sc_cols)} (ожидалось 401), total={len(feature_cols)}. "
            "Тикер не будет молча отброшен: проверь названия колонок в pipeline."
        )
    causal = causal.sort_values("bar_ts").drop_duplicates("bar_ts", keep="last").reset_index(drop=True)
    # Keep bar fields for forward path analysis, plus features and provenance.
    if perf_warning_count:
        print(f"{ticker}: подавлено предупреждений о фрагментации DataFrame: {perf_warning_count}")
    return {
        "ticker": ticker, "data": data, "dates": saved_dates, "bars": bars,
        "dataset": causal, "feature_cols": feature_cols, "base_cols": base_feature_cols,
        "sc_cols": sc_cols, "provenance": provenance,
    }

datasets = {}
load_rows = []
load_errors = []
for ticker in TICKERS:
    try:
        obj = build_ticker_dataset(ticker)
        datasets[ticker] = obj
        load_rows.append({
            "ticker": ticker, "saved_dates": len(obj["dates"]),
            "date_start": obj["dates"][0], "date_end": obj["dates"][-1],
            "dollar_bars": len(obj["bars"]), "base_features": len(obj["base_cols"]),
            "supercandle_features": len(obj["sc_cols"]), "features_total": len(obj["feature_cols"]),
        })
        print(
            f"OK {ticker}: bars={len(obj['bars']):,}; "
            f"base={len(obj['base_cols'])}; SC={len(obj['sc_cols'])}; "
            f"dates={obj['dates'][0]} → {obj['dates'][-1]}"
        )
    except Exception as e:
        message = f"{ticker}: {type(e).__name__}: {e}"
        load_errors.append(message)
        print(f"ERROR {message}")
display(pd.DataFrame(load_rows))
if load_errors:
    print("\nОшибки загрузки (кратко):")
    for message in load_errors:
        print(" -", message)
if not datasets:
    raise RuntimeError(
        "Не удалось подготовить данные ни для одного тикера. Причины выше: "
        + " | ".join(load_errors)
    )


SMLT: подавлено предупреждений о фрагментации DataFrame: 74
OK SMLT: bars=2,672; base=73; SC=401; dates=2026-09-25 → 2026-10-20
GAZP: подавлено предупреждений о фрагментации DataFrame: 74
OK GAZP: bars=9,993; base=73; SC=401; dates=2026-09-24 → 2026-10-20
MTLR: подавлено предупреждений о фрагментации DataFrame: 74
OK MTLR: bars=1,492; base=73; SC=401; dates=2026-08-28 → 2026-10-20
VTBR: подавлено предупреждений о фрагментации DataFrame: 74
OK VTBR: bars=9,482; base=73; SC=401; dates=2026-08-28 → 2026-10-20


,ticker,saved_dates,date_start,date_end,dollar_bars,base_features,supercandle_features,features_total
0,SMLT,14,2026-09-25,2026-10-20,2672,73,401,474
1,GAZP,15,2026-09-24,2026-10-20,9993,73,401,474
2,MTLR,33,2026-08-28,2026-10-20,1492,73,401,474
3,VTBR,32,2026-08-28,2026-10-20,9482,73,401,474


## 3. Свернуть 401 столбец в корневые метрики и смысловые семьи

Например, `x`, `x_ma3`, `x_z12`, `x_chg`, `x_ret` относятся к одному корневому сигналу. Семьи по названиям эвристические и нужны только для первого анализа.

In [9]:
import math, re
from collections import defaultdict

TRANSFORM_RE = re.compile(r"_(?:ma\d+|z\d+|chg|ret)$", re.IGNORECASE)
def root_feature_name(col):
    source, metric = col.split("__", 1)
    while True:
        new_metric = TRANSFORM_RE.sub("", metric)
        if new_metric == metric: break
        metric = new_metric
    return f"{source}__{metric}"
def transform_complexity(col):
    return len(TRANSFORM_RE.findall(col.split("__", 1)[1]))
def semantic_family(root_col):
    m = root_col.split("__", 1)[1].lower()
    if any(k in m for k in ("imb", "imbalance", "bid", "ask", "buy", "sell", "aggress", "b_s", "share")): return "imbalance_pressure"
    if any(k in m for k in ("spread", "liquidity", "cost", "vwap", "px_", "price", "mid", "microprice", "bps")): return "liquidity_price"
    if any(k in m for k in ("volume", "value", "turnover", "trade", "count", "qty", "notional", "amount")): return "activity_volume"
    if any(k in m for k in ("order", "depth", "level", "queue", "cancel", "add", "modify")): return "orders_depth"
    return "other"

# Inventory is shared across tickers; the loader separately validates 73 + 401 = 474 per ticker.
all_sc_cols = datasets[next(iter(datasets))]["sc_cols"]
root_members = defaultdict(list)
for col in all_sc_cols: root_members[root_feature_name(col)].append(col)
root_rows = []
for root, members in sorted(root_members.items()):
    source = root.split("__", 1)[0]
    chosen = root if root in members else sorted(members, key=lambda c:(transform_complexity(c),len(c),c))[0]
    root_rows.append({"root":root,"source":source,"family":semantic_family(root),"chosen_feature":chosen,"variants":len(members),"is_raw_root":chosen==root})
root_map = pd.DataFrame(root_rows)
print(f"401 SC columns collapsed into {root_map.root.nunique()} root metrics")
print("Root metrics by source/family:")
print(root_map.groupby(["source","family"]).size().rename("roots").to_string())
print("\nTransformation variants per root:")
print(root_map.variants.value_counts().sort_index().rename_axis("variants").to_string())


401 SC columns collapsed into 51 root metrics
Root metrics by source/family:
source      family            
obstats     activity_volume        2
            imbalance_pressure     7
            liquidity_price        7
orderstats  activity_volume        4
            imbalance_pressure    13
            liquidity_price        2
            orders_depth           2
tradestats  activity_volume        2
            imbalance_pressure    10
            other                  2

Transformation variants per root:
variants
1     16
11    35


## 4. Причинные групповые scores

Для каждого корневого признака отклонение от rolling median нормируется на прошлый 95-й перцентиль отклонений. Все окна сдвинуты на один бар. Затем групповой score — среднее верхних 20% root-scores; событие — превышение прошлого 95-го или 99-го перцентиля группового score.

In [10]:
def find_ohlc_columns(bars):
    lower = {str(c).lower():c for c in bars.columns}
    out = {k:next((lower[x] for x in names if x in lower),None) for k,names in {
        "open":("open","o"),"high":("high","h"),"low":("low","l"),"close":("close","c")}.items()}
    missing = [k for k,v in out.items() if v is None]
    if missing: raise RuntimeError(f"OHLC columns missing {missing}; columns={list(bars.columns)}")
    return out

def root_scores_for_ticker(obj):
    ds = obj["dataset"].reset_index(drop=True)
    out = {}
    for _, r in root_map.iterrows():
        col = r["chosen_feature"]
        if col not in ds: continue
        x = pd.to_numeric(ds[col],errors="coerce").replace([np.inf,-np.inf],np.nan)
        med = x.rolling(ROLLING_WINDOW,min_periods=MIN_HISTORY).median().shift(1)
        dev = (x-med).abs()
        scale = dev.rolling(ROLLING_WINDOW,min_periods=MIN_HISTORY).quantile(.95).shift(1)
        out[r["root"]] = dev / scale.where(scale>0)
    return pd.DataFrame(out)

def top_fraction_mean(frame, fraction=.20):
    if frame.shape[1]==0: return pd.Series(np.nan,index=frame.index)
    a=frame.to_numpy(dtype=float).copy(); a[~np.isfinite(a)]=-np.inf; a.sort(axis=1)
    k=max(1,int(math.ceil(frame.shape[1]*fraction))); top=a[:,-k:]
    ok=np.isfinite(top); sums=np.where(ok,top,0).sum(axis=1); counts=ok.sum(axis=1)
    return pd.Series(np.divide(sums,counts,out=np.full(len(frame),np.nan),where=counts>0),index=frame.index)

def group_scores_for_ticker(obj):
    roots=root_scores_for_ticker(obj); meta=root_map.set_index("root"); groups={}
    for source in ("tradestats","obstats","orderstats"):
        cols=[c for c in roots if meta.loc[c,"source"]==source]
        if cols: groups[f"source::{source}"]=top_fraction_mean(roots[cols])
    for (source,family),part in root_map.groupby(["source","family"]):
        cols=[c for c in part.root.tolist() if c in roots]
        if len(cols)>=2: groups[f"source_family::{source}::{family}"]=top_fraction_mean(roots[cols])
    scores=pd.DataFrame(groups)
    flags={}; thresholds={}
    for col in scores:
        for q in GROUP_QUANTILES:
            th=scores[col].rolling(ROLLING_WINDOW,min_periods=MIN_HISTORY).quantile(q).shift(1)
            thresholds[(col,q)]=th; flags[(col,q)]=scores[col].gt(th)&th.notna()&scores[col].notna()
    return scores,thresholds,flags

group_results={}
for ticker,obj in datasets.items():
    scores,thresholds,flags=group_scores_for_ticker(obj)
    group_results[ticker]={"scores":scores,"thresholds":thresholds,"flags":flags}
    print(f"{ticker}: {scores.shape[1]} source/family groups; {len(scores):,} bars")


SMLT: 13 source/family groups; 2,672 bars
GAZP: 13 source/family groups; 9,993 bars
MTLR: 13 source/family groups; 1,492 bars
VTBR: 13 source/family groups; 9,482 bars


## 5. Эпизоды и сопоставленные контрольные бары

Соседние срабатывания сворачиваются в один эпизод. Контроли подбираются по квинтилю прошлой волатильности; доходности, MFE и MAE считаются в процентных пунктах от close бара события. Bootstrap CI ориентировочен: он не устраняет всю временную зависимость.

In [11]:
HORIZONS=(1,3,5,10,20); BOOTSTRAP_REPS=400

def episode_starts(mask,min_gap=EPISODE_GAP_BARS):
    starts=[]; last=-10**9
    for i in np.flatnonzero(np.asarray(mask,dtype=bool)):
        if int(i)-last>=min_gap: starts.append(int(i)); last=int(i)
    return starts

def bootstrap_ci(values,reps=BOOTSTRAP_REPS,seed=RANDOM_STATE):
    x=np.asarray(values,dtype=float); x=x[np.isfinite(x)]
    if len(x)<10: return (np.nan,np.nan)
    rng=np.random.default_rng(seed); sims=np.empty(reps)
    for b in range(reps): sims[b]=rng.choice(x,size=len(x),replace=True).mean()
    return tuple(np.quantile(sims,[.025,.975]))

def path_metrics(close,high,low,i):
    p0=close[i]
    if not np.isfinite(p0) or p0==0: return {}
    out={}
    for h in HORIZONS:
        j=i+h
        if j>=len(close): continue
        out[f"ret_{h}"]=(close[j]/p0-1)*100
        hi=high[i+1:j+1]; lo=low[i+1:j+1]
        out[f"mfe_{h}"]=(np.nanmax(hi)/p0-1)*100 if np.isfinite(hi).any() else np.nan
        out[f"mae_{h}"]=(np.nanmin(lo)/p0-1)*100 if np.isfinite(lo).any() else np.nan
        out[f"abs_ret_{h}"]=abs(out[f"ret_{h}"])
    return out

def analyze_group(ticker,obj,group_key,q,flag,score):
    bars=obj["bars"].reset_index(drop=True); c=find_ohlc_columns(bars)
    close=pd.to_numeric(bars[c["close"]],errors="coerce").to_numpy(float)
    high=pd.to_numeric(bars[c["high"]],errors="coerce").to_numpy(float)
    low=pd.to_numeric(bars[c["low"]],errors="coerce").to_numpy(float)
    n=len(close); end=n-max(HORIZONS)-1
    raw_idx=np.flatnonzero(np.asarray(flag,dtype=bool)); raw_idx=raw_idx[raw_idx<end]
    starts=[i for i in episode_starts(flag) if i<end]
    if not starts: return []
    tv=pd.Series(close).pct_change().rolling(50,min_periods=20).std().shift(1)
    try: bucket=pd.qcut(tv.rank(method="first"),5,labels=False,duplicates="drop")
    except Exception: bucket=pd.Series(0,index=tv.index)
    event_mask=np.asarray(flag,dtype=bool)
    excluded = event_mask.copy()
    for e in starts:
        excluded[max(0, e-EPISODE_GAP_BARS+1):min(n, e+EPISODE_GAP_BARS)] = True
    candidates=[j for j in range(MIN_HISTORY,max(MIN_HISTORY,end)) if np.isfinite(close[j]) and pd.notna(bucket.iloc[j]) and not excluded[j]]
    candidates_by_bucket = {}
    for j in candidates:
        candidates_by_bucket.setdefault(int(bucket.iloc[j]), []).append(j)
    # Stable seed across Python sessions.
    stable=sum(ord(ch) for ch in f"{ticker}|{group_key}|{q}")
    rng=np.random.default_rng(RANDOM_STATE+stable); records=[]
    for ei in starts:
        if pd.isna(bucket.iloc[ei]): continue
        same=candidates_by_bucket.get(int(bucket.iloc[ei]), [])
        if not same: continue
        chosen=rng.choice(same,size=min(MAX_CONTROLS_PER_EVENT,len(same)),replace=False)
        ev=path_metrics(close,high,low,ei); ctr=[path_metrics(close,high,low,int(j)) for j in chosen]
        if not ev: continue
        row={"ticker":ticker,"group_type":group_key.split("::",1)[0],"group":group_key.split("::",1)[1],"quantile":q,
             "bars":n,"raw_event_bars":len(raw_idx),"event_bar_pct":100*len(raw_idx)/max(n,1),"episodes":len(starts),"event_idx":ei}
        for h in HORIZONS:
            key=f"ret_{h}"
            if key not in ev: continue
            cv=np.array([m.get(key,np.nan) for m in ctr],float); cv=cv[np.isfinite(cv)]
            if not len(cv): continue
            row[f"paired_diff_{h}"]=ev[key]-cv.mean()
            for metric in (key,f"mfe_{h}",f"mae_{h}",f"abs_ret_{h}"):
                vv=ev.get(metric,np.nan); cc=np.array([m.get(metric,np.nan) for m in ctr],float); cc=cc[np.isfinite(cc)]
                row[f"event_{metric}"]=vv; row[f"control_{metric}"]=float(cc.mean()) if len(cc) else np.nan
        records.append(row)
    return records

all_rows=[]
for ticker,obj in datasets.items():
    res=group_results[ticker]
    for (group_key,q),flag in res["flags"].items():
        all_rows.extend(analyze_group(ticker,obj,group_key,q,flag,res["scores"][group_key]))
    print(f"{ticker}: matched event episodes={sum(r['ticker']==ticker for r in all_rows):,} cumulative across groups")
event_level=pd.DataFrame(all_rows)
if event_level.empty: raise RuntimeError("No episodes with matched controls; inspect thresholds or data coverage.")


SMLT: matched event episodes=1,088 cumulative across groups
GAZP: matched event episodes=3,896 cumulative across groups
MTLR: matched event episodes=661 cumulative across groups
VTBR: matched event episodes=3,608 cumulative across groups


## 6. Итог — один компактный CSV

Одна строка на тикер × группа × порог × горизонт. Не выбирать лучший результат только по этой выборке; использовать её для формирования заранее заданных гипотез.

In [14]:
summary_rows=[]
for keys,g in event_level.groupby(["ticker","group_type","group","quantile"],dropna=False):
    ticker,group_type,group,q=keys
    for h in HORIZONS:
        dc=f"paired_diff_{h}"; ec=f"event_ret_{h}"; cc=f"control_ret_{h}"
        if dc not in g or g[dc].notna().sum()==0: continue
        paired=pd.to_numeric(g[dc],errors="coerce").dropna().to_numpy(float)
        lo,hi=bootstrap_ci(paired,seed=RANDOM_STATE+int(q*1000)+h+sum(map(ord,str(group))))
        summary_rows.append({
            "ticker":ticker,"group_type":group_type,"group":group,"quantile":q,"horizon_bars":h,
            "bars":int(g.bars.iloc[0]),"raw_event_bars":int(g.raw_event_bars.iloc[0]),
            "event_bar_pct":float(g.event_bar_pct.iloc[0]),"episodes":int(g.episodes.iloc[0]),
            "episodes_with_controls":int(g[dc].notna().sum()),
            "event_mean_ret_pct":float(pd.to_numeric(g[ec],errors="coerce").mean()),
            "control_mean_ret_pct":float(pd.to_numeric(g[cc],errors="coerce").mean()),
            "paired_mean_diff_pp":float(paired.mean()),"paired_median_diff_pp":float(np.median(paired)),
            "bootstrap_ci_low_pp":lo,"bootstrap_ci_high_pp":hi,
            "event_mean_abs_ret_pct":float(pd.to_numeric(g[f"event_abs_ret_{h}"],errors="coerce").mean()),
            "control_mean_abs_ret_pct":float(pd.to_numeric(g[f"control_abs_ret_{h}"],errors="coerce").mean()),
            "event_mean_mfe_pct":float(pd.to_numeric(g[f"event_mfe_{h}"],errors="coerce").mean()),
            "event_mean_mae_pct":float(pd.to_numeric(g[f"event_mae_{h}"],errors="coerce").mean()),
        })
summary=pd.DataFrame(summary_rows)
if summary.empty: raise RuntimeError("No horizon summary rows generated.")
summary["ci_excludes_zero"]=summary.bootstrap_ci_low_pp.notna() & ((summary.bootstrap_ci_low_pp>0)|(summary.bootstrap_ci_high_pp<0))
summary=summary.sort_values(["ticker","group_type","group","quantile","horizon_bars"]).reset_index(drop=True)
OUTPUT_FILE=Path("research/supercandles/mark_2/results/SC_02_group_research_summary.csv"); OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUTPUT_FILE,index=False)
print(f"Saved ONE compact file: {OUTPUT_FILE.resolve()} ({len(summary):,} rows; {OUTPUT_FILE.stat().st_size/1024:.1f} KB)")
summary

Saved ONE compact file: C:\Users\Сергей Кравцов\SC_02_group_research_summary.csv (520 rows; 146.7 KB)


,ticker,group_type,group,quantile,horizon_bars,bars,raw_event_bars,event_bar_pct,episodes,episodes_with_controls,...,control_mean_ret_pct,paired_mean_diff_pp,paired_median_diff_pp,bootstrap_ci_low_pp,bootstrap_ci_high_pp,event_mean_abs_ret_pct,control_mean_abs_ret_pct,event_mean_mfe_pct,event_mean_mae_pct,ci_excludes_zero
0,GAZP,source,obstats,0.95,1,9993,544,5.443811,211,211,...,0.002438,0.040054,0.002936,-0.005036,0.110140,0.104435,0.061647,0.099204,-0.053312,False
1,GAZP,source,obstats,0.95,3,9993,544,5.443811,211,211,...,-0.000699,0.052794,0.023666,0.000716,0.131763,0.144452,0.106980,0.146672,-0.088323,True
2,GAZP,source,obstats,0.95,5,9993,544,5.443811,211,211,...,0.004317,0.089087,0.034277,0.009466,0.195909,0.209431,0.133237,0.218057,-0.115924,True
3,GAZP,source,obstats,0.95,10,9993,544,5.443811,211,211,...,0.023755,0.094410,0.016847,0.011389,0.193675,0.270752,0.199373,0.279239,-0.158802,True
4,GAZP,source,obstats,0.95,20,9993,544,5.443811,211,211,...,0.017064,0.142471,0.110265,0.049430,0.260292,0.351086,0.279431,0.371920,-0.215981,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
515,VTBR,source_family,tradestats::other,0.99,1,9482,107,1.128454,54,54,...,0.020314,-0.054766,-0.030131,-0.093181,-0.019856,0.096198,0.091891,0.093928,-0.102773,True
516,VTBR,source_family,tradestats::other,0.99,3,9482,107,1.128454,54,54,...,0.018285,-0.047221,-0.025785,-0.110715,0.034543,0.173804,0.135836,0.133335,-0.179545,False
517,VTBR,source_family,tradestats::other,0.99,5,9482,107,1.128454,54,54,...,-0.001320,-0.028250,-0.042961,-0.109754,0.062574,0.227026,0.208412,0.168122,-0.219941,False
518,VTBR,source_family,tradestats::other,0.99,10,9482,107,1.128454,54,54,...,0.041960,-0.112099,-0.211173,-0.214101,-0.004229,0.290217,0.254330,0.255815,-0.316062,True


## 7. Как интерпретировать

- Смотри `event_bar_pct` вместе с `episodes`: частые бары могут быть лишь кластеризацией одного эпизода.
- Сравнивай отдельно три источника и `source × family`, оба тикера и оба порога.
- `paired_mean_diff_pp` — ассоциация с будущей доходностью, не PnL стратегии. Для торговли отдельно нужны реальное время доступности, исполнимый вход, bid/ask, комиссии и проскальзывание.
- Не запускаем 401 отдельных теста и не создаём пачку промежуточных файлов.

In [18]:
summary.sort_values('paired_mean_diff_pp', ascending=False).head(10)

,ticker,group_type,group,quantile,horizon_bars,bars,raw_event_bars,event_bar_pct,episodes,episodes_with_controls,...,control_mean_ret_pct,paired_mean_diff_pp,paired_median_diff_pp,bootstrap_ci_low_pp,bootstrap_ci_high_pp,event_mean_abs_ret_pct,control_mean_abs_ret_pct,event_mean_mfe_pct,event_mean_mae_pct,ci_excludes_zero
139,MTLR,source,obstats,0.99,20,1492,9,0.603217,6,6,...,-0.405553,2.052105,1.491776,NaN,NaN,1.743428,1.886904,2.336001,-1.189762,False
189,MTLR,source_family,obstats::liquidity_price,0.99,20,1492,9,0.603217,6,6,...,0.354227,1.256078,1.275063,NaN,NaN,1.707181,1.809199,2.215511,-0.808205,False
179,MTLR,source_family,obstats::imbalance_pressure,0.99,20,1492,20,1.340483,14,14,...,0.089344,1.142507,0.793471,-0.361805,2.728104,1.839732,1.477491,2.386764,-0.791400,False
269,SMLT,source,obstats,0.99,20,2672,26,0.973054,14,14,...,-0.120540,0.860632,0.881795,0.365565,1.293487,0.924572,0.963471,1.439012,-0.472455,True
319,SMLT,source_family,obstats::liquidity_price,0.99,20,2672,28,1.047904,16,16,...,-0.300731,0.796514,0.905147,0.218768,1.323212,0.958892,0.930104,1.420396,-0.644254,True
318,SMLT,source_family,obstats::liquidity_price,0.99,10,2672,28,1.047904,16,16,...,-0.226863,0.778022,0.757366,0.319645,1.231812,0.921461,0.803770,1.027114,-0.507843,True
299,SMLT,source_family,obstats::activity_volume,0.99,20,2672,40,1.497006,17,17,...,-0.192953,0.614243,0.951725,0.061507,1.147824,0.728082,1.184754,1.036337,-0.547664,True
268,SMLT,source,obstats,0.99,10,2672,26,0.973054,14,14,...,0.072483,0.519414,0.557023,0.155827,0.843769,0.808227,0.717470,1.015104,-0.391260,True
138,MTLR,source,obstats,0.99,10,1492,9,0.603217,6,6,...,-0.436417,0.491161,-0.490831,NaN,NaN,1.065607,0.949396,1.005480,-1.117058,False
184,MTLR,source_family,obstats::liquidity_price,0.95,20,1492,42,2.815013,27,27,...,0.300722,0.490046,0.896464,-0.342160,1.196339,1.770421,1.641635,2.056880,-1.174710,False


In [19]:
summary.sort_values('paired_mean_diff_pp').head(10)

,ticker,group_type,group,quantile,horizon_bars,bars,raw_event_bars,event_bar_pct,episodes,episodes_with_controls,...,control_mean_ret_pct,paired_mean_diff_pp,paired_median_diff_pp,bootstrap_ci_low_pp,bootstrap_ci_high_pp,event_mean_abs_ret_pct,control_mean_abs_ret_pct,event_mean_mfe_pct,event_mean_mae_pct,ci_excludes_zero
169,MTLR,source_family,obstats::activity_volume,0.99,20,1492,13,0.871314,6,6,...,1.822166,-2.262855,-3.347175,NaN,NaN,2.147958,2.631388,1.858443,-2.199071,False
254,MTLR,source_family,tradestats::other,0.95,20,1492,56,3.753351,39,39,...,0.640228,-1.038247,-0.810317,-1.765533,-0.353456,1.558950,1.555639,1.219643,-1.585717,True
234,MTLR,source_family,tradestats::activity_volume,0.95,20,1492,70,4.691689,49,49,...,0.713716,-0.985565,-1.320467,-1.626759,-0.380274,1.407686,1.717001,1.207965,-1.630236,True
238,MTLR,source_family,tradestats::activity_volume,0.99,10,1492,12,0.804290,10,10,...,0.232467,-0.663218,-0.265726,-1.381734,0.167481,0.996302,1.224956,0.554650,-1.169697,False
253,MTLR,source_family,tradestats::other,0.95,10,1492,56,3.753351,39,39,...,0.426470,-0.633047,-1.002288,-1.096927,-0.171593,0.865576,1.184984,0.843979,-0.974107,True
239,MTLR,source_family,tradestats::activity_volume,0.99,20,1492,12,0.804290,10,10,...,0.474291,-0.611104,-0.626999,-1.506207,0.294425,1.084128,1.685610,0.914871,-1.470991,False
237,MTLR,source_family,tradestats::activity_volume,0.99,5,1492,12,0.804290,10,10,...,0.132451,-0.586392,-0.729084,-0.941326,-0.166617,0.564438,0.813177,0.463443,-0.750637,True
167,MTLR,source_family,obstats::activity_volume,0.99,5,1492,13,0.871314,6,6,...,0.736197,-0.574039,-0.523428,NaN,NaN,0.521592,1.393709,0.714248,-0.464653,False
233,MTLR,source_family,tradestats::activity_volume,0.95,10,1492,70,4.691689,49,49,...,0.302386,-0.567109,-0.568661,-0.944185,-0.189770,0.982790,1.045156,0.811270,-1.065431,True
218,MTLR,source_family,orderstats::liquidity_price,0.99,10,1492,18,1.206434,13,13,...,0.219912,-0.556771,-0.433941,-1.243828,0.117939,0.910316,0.999723,0.930317,-1.051077,False
